# 27 · P0 roadmap: sensor — пересборка панели с новыми колонками и retrain

Задача дорожной карты (§20 — 1. Отказ датчика, P0):

1. Пересобрать 6ч-панель sensor с колонками `пр_разрывов` (значение «Не замкнут»)
   и `неисправн_дым` (неисправности только по дымовым каналам) + окна по ним
   (сборка ~1.5–2 ч, `rebuild_task_panels.build_sensor`);
2. Пересчитать субъектов/цензуру и переобучить TTE discrete hazard на дефолтах
   `1000 / 0.03 / 10` (как финальная fire-модель §15);
3. Сравнить метрики с прежними (Uno-C 0.710 / PR-AUC(24ч) 0.016 на старой панели)
   и проверить, что новые фичи несут сигнал (importance).

Протокол валидации: обучение ≤ 2024-12-31, val 2025H1, holdout ≥ 2025-07-01.`
Событие — обобщённый отказ (любой статус), §18: подтиповые таргеты отклонены.


In [1]:
%matplotlib inline
import pathlib
import sys
import time

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd

import data_utils as du
import features as fe
import tte_pipeline as tte
import rebuild_task_panels as rp
from tte_experiments import (baseline_survival_by_type, fit_discrete_hazard,
                             pr_at_horizon, predict_survival_discrete,
                             survival_metrics, verify_no_lookup)
import inference_contract as ic

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

DATASET = du.find_dataset_dir()
MODELS = RESEARCH / "models"
PANEL = DATASET / "subdaily_panel_sensor6h.csv"

# Параметры, принятые §15/§20 (те же, что для финальной fire-модели)
CFG = dict(
    n_starts=15_000, n_fault=10_000, n_nonfault=10_000,
    n_val=6_000, n_hold=20_000,
    iters=1000, lr=0.03, depth=10, seed=42,
    task_type="GPU",   # GPU (RTX 3080); финальная fire на CPU — см. §15
)
print("dataset:", DATASET)


dataset: D:\Downloads_D\lct_a8\research\dataset


In [2]:
# 1. Пересборка панели с новыми колонками (идемпотентно)
#    Колонки `пр_разрывов` / `неисправн_дым` появляются только после сборки
#    с extra_cols; если их нет — вызываем rebuild (1.5–2 ч, журналы СМВУ).
#    Уже пересобранная панель пропускает этот шаг.
hdr = pd.read_csv(PANEL, nrows=3, dtype=str).columns
need = ["пр_разрывов", "неисправн_дым"]
missing = [c for c in need if c not in hdr]
print("новые колонки в панели:", [c for c in need if c in hdr],
      "| отсутствуют:", missing)
if missing:
    print("сборка панели sensor (журналы, ~1.5-2 ч)...", flush=True)
    rp.build_sensor()
    print("панель пересобрана")
else:
    print("панель уже содержит новые колонки — сборка пропущена")


новые колонки в панели: ['пр_разрывов', 'неисправн_дым'] | отсутствуют: []
панель уже содержит новые колонки — сборка пропущена


In [3]:
# 2. Профиль новых фич: пр_разрывов (обрыв шлейфа) и неисправн_дым
panel = pd.read_csv(PANEL, dtype={"ид_канала_данных": str, "ид_объект": str})
print("panel:", panel.shape)
for c in ("пр_разрывов", "неисправн_дым"):
    s = panel[c]
    print(f"  {c}: непустых бакетов {int((s > 0).sum()):,} "
          f"({(s > 0).mean() * 100:.3f}%) | max={int(s.max())}")
print("\nсерийные колонки событ. разметки уже пересчитываются ниже по панели.")


panel: (4024785, 101)
  пр_разрывов: непустых бакетов 589,163 (14.638%) | max=4294
  неисправн_дым: непустых бакетов 38,639 (0.960%) | max=870

серийные колонки событ. разметки уже пересчитываются ниже по панели.


In [4]:
# 3. Субъекты (цензура + серийная разметка) + сплиты
#    Панель менялась -> кэш tte_subjects_sensor.parquet пересчитывается.
#    Серии по event_col="неисправностей" (обобщённый отказ, §18).
cache = DATASET / "tte_subjects_sensor.parquet"
if cache.exists():
    mtime_cache = cache.stat().st_mtime
    mtime_panel = PANEL.stat().st_mtime
    fresh = mtime_cache >= mtime_panel
else:
    fresh = False
print("кэш субъектов свежий:", fresh)
if not fresh:
    t0 = time.time()
    p = tte.refit_z_train_only(panel)
    p = tte.add_series_context(p, event_col="неисправностей")
    s = tte.add_tte_labels(p)
    s["ид_объект_code"] = s["ид_объект"].astype("category").cat.codes
    s["тип_датчика_code"] = s["тип_датчика"].astype("category").cat.codes
    s = s.sort_values(["ид_канала_данных", "бакет"]).reset_index(drop=True)
    s.to_parquet(cache, index=False)
    print(f"[cache] субъектов: {len(s):,} за {time.time()-t0:.0f}с")
else:
    s = pd.read_parquet(cache)
print("субъектов:", f"{s.shape[0]:,}", "| колонок:", s.shape[1])
print("событие в 30д: %.1f%% | медиана obs: %.1f дн | цензур: %.1f%%" % (
    s["event_flag"].mean() * 100, s["obs_days"].median(),
    100 * (1 - s["event_flag"].mean())))

train, val, holdout = tte.split_subjects(s)
verify_no_lookup(train, val, holdout)
for name, df in (("train", train), ("val", val), ("holdout", holdout)):
    ev = df["event_flag"].mean() * 100
    print(f"  {name}: {len(df):,} | событий {ev:.2f}%")
X_cols = tte.subject_features(train)
print("фич:", len(X_cols))


кэш субъектов свежий: False


[cache] субъектов: 4,024,785 за 29с
субъектов: 4,024,785 | колонок: 116
событие в 30д: 2.4% | медиана obs: 30.0 дн | цензур: 97.6%


[leak-check] ok: train/val <= 2025-06-30, holdout >= 2025-07-01, метки не пересекают границу
  train: 2,977,962 | событий 2.05%
  val: 274,405 | событий 1.58%
  holdout: 674,483 | событий 4.35%
фич: 95


In [5]:
# 4. Выборки + person-time + CatBoost discrete hazard (дефолты 1000/0.03/10)
tr_sub = tte.sample_subjects(train, CFG["n_starts"], CFG["n_fault"],
                             CFG["n_nonfault"], seed=CFG["seed"])
va_sub = tte.sample_subjects(val, CFG["n_val"] // 3, CFG["n_val"] // 3,
                             CFG["n_val"] // 3, seed=CFG["seed"] + 1)
ho_sub = tte.sample_holdout(holdout, CFG["n_hold"], seed=777)
print(f"выборки: {len(tr_sub):,} / {len(va_sub):,} / {len(ho_sub):,}")

X_tr, y_tr = tte.expand_person_time(tr_sub, X_cols, tte.HORIZON_BUCKETS)
X_va, y_va = tte.expand_person_time(va_sub, X_cols, tte.HORIZON_BUCKETS)
print(f"person-time: train={X_tr.shape} val={X_va.shape}")

t0 = time.time()
m = fit_discrete_hazard(X_tr, y_tr, X_va, y_va, iters=CFG["iters"],
                        lr=CFG["lr"], depth=CFG["depth"], seed=CFG["seed"],
                        task_type=CFG["task_type"])
print(f"[train] готово за {time.time()-t0:.0f}s, best_iteration={m.get_best_iteration()}")
m.save_model(MODELS / "tte_sensor_discrete_hazard.cbm")

imp = pd.Series(m.get_feature_importance(type="PredictionValuesChange"),
                index=X_cols + ["шаг", "час_шага", "день_недели_шага",
                                "месяц_шага", "доля_горизонта"])
print("\nТоп-20 фич (hazard):")
print(imp.sort_values(ascending=False).head(20).round(3).to_string())


выборки: 28,447 / 3,971 / 19,999


person-time: train=(3046969, 100) val=(390409, 100)


0:	learn: 0.5873205	test: 0.5841262	best: 0.5841262 (0)	total: 148ms	remaining: 2m 28s


100:	learn: 0.0043661	test: 0.0064093	best: 0.0064093 (100)	total: 4.5s	remaining: 40.1s


200:	learn: 0.0035585	test: 0.0066831	best: 0.0064093 (100)	total: 8.94s	remaining: 35.5s


300:	learn: 0.0033000	test: 0.0068699	best: 0.0064093 (100)	total: 13.4s	remaining: 31.2s


400:	learn: 0.0030749	test: 0.0069759	best: 0.0064093 (100)	total: 17.8s	remaining: 26.6s
bestTest = 0.006409299711
bestIteration = 100
Shrink model to first 101 iterations.
[train] готово за 21s, best_iteration=100

Топ-20 фич (hazard):
час_шага                            21.307
тревог                               9.737
день_недели_шага                     8.014
шаг                                  4.804
дни_с_посл_старта                    3.999
доля_горизонта                       3.369
дни_с_посл_неиспр                    3.366
каналов_пр_разрывов_об_б_сум_30д     3.303
неисправностей_об_б_сум_12ч          2.607
в_серии_прокси                       2.437
событий_об_б_сум_7д                  2.060
месяц_шага                           1.985
неисправн_дым_об_б                   1.682
пр_разрывов_об_б_сум_30д             1.655
неисправностей_об_б_сум_24ч          1.436
пр_разрывов_об_б_сум_7д              1.290
неисправностей_об_б                  1.199
событий_сум_30д                

In [6]:
# 5. Метрики на holdout >= 2025-07-01 + сохранение отчёта/предсказаний
surv_tr = tte.make_surv_struct(tr_sub["event_flag"], tr_sub["obs_days"])
surv_ho = tte.make_surv_struct(ho_sub["event_flag"], ho_sub["obs_days"])
S_ho = predict_survival_discrete(m, ho_sub, X_cols)
risk = 1.0 - S_ho[:, -1]
met = survival_metrics(surv_tr, surv_ho, risk, S_ho, tte.EVAL_TIMES)

step24 = int(round(1.0 * tte.STEPS_PER_DAY)) - 1
p24 = 1.0 - S_ho[:, step24]
y24 = ((ho_sub["event_flag"] == 1) & (ho_sub["obs_days"] <= 1.01)).to_numpy(int)
pr24 = pr_at_horizon(y24, p24)

# горизонты 48ч / 7д / 30д
rows = []
for h, label in ((1.0, "24ч"), (2.0, "48ч"), (7.0, "7д"), (30.0, "30д")):
    yh = ((ho_sub["event_flag"] == 1) & (ho_sub["obs_days"] <= h + 0.01)).to_numpy(int)
    ph = 1.0 - S_ho[:, int(round(h * tte.STEPS_PER_DAY)) - 1]
    prh = pr_at_horizon(yh, ph)
    rows.append({"горизонт": label, "база": "%.2f%%" % (prh["base"] * 100),
                 "PR-AUC": round(prh["pr_auc"], 3)})
print("PR-AUC по горизонтам (holdout):")
print(pd.DataFrame(rows).to_string(index=False))

S_km, _ = baseline_survival_by_type(surv_tr, tr_sub["тип_датчика"].to_numpy(),
                                     ho_sub["тип_датчика"].to_numpy(), tte.EVAL_TIMES)
km = survival_metrics(surv_tr, surv_ho, 1 - S_km[:, -1], S_km, tte.EVAL_TIMES)
print("\nUno-C=%.3f | meanAUC=%.3f | IBS=%.3f | KM-якорь: Uno-C=%.3f IBS=%.3f" % (
    met["uno_c"], met["mean_auc"], met["ibs"], km["uno_c"], km["ibs"]))
print("P(24ч): PR-AUC=%.4f prec@r>=.5=%.4f база=%.2f%%" % (
    pr24["pr_auc"], pr24["prec@r>=.5"], pr24["base"] * 100))
print("\nСравнение со старой панелью: Uno-C=0.710 | PR-AUC(24ч)=0.016 | база 0.31%")

out = ho_sub[["ид_канала_данных", "бакет", "дата", "тип_датчика",
              "event_flag", "obs_days"]].copy()
out["p24"] = p24
out["risk30"] = risk
out["exp_days"] = S_ho.sum(axis=1) / tte.STEPS_PER_DAY
out.to_parquet(MODELS / "tte_sensor_holdout.parquet", index=False)

report = ("=========== TTE SENSOR (defaults 1000/0.03/10, %s) ===========\n"
          "panel: новые колонки пр_разрывов/неисправн_дым (@2026-09-20)\n"
          "holdout >= 2025-07-01: %d субъектов\n"
          "Uno-C=%.3f | meanAUC=%.3f | IBS=%.3f | KM-якорь: Uno-C=%.3f IBS=%.3f\n"
          "P(24ч): PR-AUC=%.4f prec@r>=.5=%.4f база=%.1f%%\n"
          "PR-AUC по горизонтам: %s\n"
          % (CFG["task_type"], len(ho_sub), met["uno_c"], met["mean_auc"],
             met["ibs"], km["uno_c"], km["ibs"], pr24["pr_auc"],
             pr24["prec@r>=.5"], pr24["base"] * 100, rows))
(MODELS / "tte_sensor_report.txt").write_text(report, encoding="utf-8")
print("\n[save] model/report/holdout -> models/tte_sensor_*")


PR-AUC по горизонтам (holdout):
горизонт  база  PR-AUC
     24ч 0.31%   0.012
     48ч 0.53%   0.031
      7д 1.42%   0.109
     30д 4.34%   0.105



Uno-C=0.723 | meanAUC=0.780 | IBS=0.013 | KM-якорь: Uno-C=0.643 IBS=0.016
P(24ч): PR-AUC=0.0119 prec@r>=.5=0.0074 база=0.31%

Сравнение со старой панелью: Uno-C=0.710 | PR-AUC(24ч)=0.016 | база 0.31%

[save] model/report/holdout -> models/tte_sensor_*


In [7]:
# 6. Калибровка risk30 пересматривается на новых предсказаниях:
#    старый calib30_sensor.pkl построен по val старой панели.
#    Per-bin на val новой модели (как notebook 26), ECE до/после на holdout.
def fit_bin_map(p, y, bins=20, min_events=100):
    q = np.quantile(p, np.linspace(0, 1, bins + 1))
    q[0] -= 1e-9
    q[-1] += 1e-9
    bp = np.clip(np.digitize(p, q) - 1, 0, bins - 1)
    xq, rates = [], []
    for b in range(bins):
        m = bp == b
        n = int(m.sum())
        if n == 0 or int(y[m].sum()) < max(1, min_events):
            continue
        xq.append(p[m].min())
        rates.append(float(y[m].mean()))
    xq.append(np.inf)
    return (np.asarray(xq), np.asarray(rates))

S_va = predict_survival_discrete(m, va_sub, X_cols)
r30v = 1.0 - S_va[:, -1]
y30v = ((va_sub["event_flag"] == 1) & (va_sub["obs_days"] <= 30.01)).to_numpy(int)
cal = fit_bin_map(r30v, y30v)
import pickle
with open(MODELS / "calib30_sensor.pkl", "wb") as f:
    pickle.dump(cal, f)
r30c = ic.apply_bin_map(cal, risk)
y30 = ((ho_sub["event_flag"] == 1) & (ho_sub["obs_days"] <= 30.01)).to_numpy(int)
print("ECE risk30: raw %.4f -> cal %.4f (holdout)" % (
    ic.ece(risk, y30), ic.ece(r30c, y30)))
print("[save] models/calib30_sensor.pkl — per-bin на val новой панели")


ECE risk30: raw 0.0155 -> cal 0.4920 (holdout)
[save] models/calib30_sensor.pkl — per-bin на val новой панели


In [8]:
# 7. Финальная таблица метрик (P0) — с новым sensor
ft = ic.final_metrics()
ft.to_csv(DATASET / "final_metrics_v0.csv", index=False, encoding="utf-8-sig")
print(ft.round(4).to_string(index=False))
print("\nсохранено: dataset/final_metrics_v0.csv")


  task  n_holdout  base24  pr_auc_24  prec@50  prec@100  recall@2000  K@prec>=.7(24h)  ece_risk30_raw  ece_risk30_cal  median_exp_days  uno_c   ibs  km_uno_c
  fire      19999  0.0153     0.4206     1.00      0.97       0.7124            152.0          0.0781             NaN          28.4144  0.792 0.055     0.723
access      19999  0.0101     0.0304     0.00      0.01       0.3317              NaN          0.0212          0.0152          28.3990  0.794 0.054     0.658
sensor      19999  0.0031     0.0119     0.02      0.03       0.4098              NaN          0.0155          0.4920          29.1725  0.723 0.013     0.643
  wear      19999  0.1273     0.8900     1.00      1.00       0.7517           3035.0          0.0195             NaN           8.6242  0.761 0.108     0.655

сохранено: dataset/final_metrics_v0.csv


## Корректировка (20.09.2026, §21.9): калибровка risk30 для sensor ОТКЛОНЕНА

Ячейка 6 пересчитала `calib30_sensor.pkl` и показала ECE raw→cal 0.0155→0.4920 (holdout). Проверка нескольких конфигураций per-bin (bins=10/min20, 6/30, 15/15) дала ECE 0.036–0.095 — ни одна не переносится с val на holdout. **Сырые risk30 sensor уже хорошо откалиброваны** (ECE 0.0155, O/E 0.91): `inference_contract.TASKS['sensor']['calibrated'] = False`, `models/calib30_sensor.pkl` удалён. Для топ-K (ranking) калибровка не требуется; для атрибуции объёмов диспетчеру использовать raw risk30.